In [ ]:
import pandas as pd
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional, List 

class OperationType(str, Enum):
    RENAME = "rename"
    CAST = "cast" 
    TO_DATETIME = "to_datetime" 
    FILLNA = "fillna" 
    DROP_COLUMNS = "drop_columns" 
    FEATURE_ENGINEERING = "feature_engineering"

class TransformationStep(BaseModel):
    type: OperationType = Field(description="Type of transformation operation")
    source_column: Optional[str] = Field(default=None, description="Original column name in the dataset")
    source_dtype: Optional[str] = Field(default=None, description="Original column datatype")
    target_column: Optional[str] = Field(default=None, description="Column name after transformation")
    target_dtype: Optional[str] = Field(default=None, description="Expected datatype after transformation")
    source_columns: Optional[List[str]] = Field(default=None, description="Multiple columns used for feature engineering")
    formula: Optional[str] = Field(default=None, description="Formula used for feature engineering")
    fill_value: Optional[str] = Field(default=None, description="Value used for missing value imputation")
    allow_if_missing: Optional[bool] = Field(default=False, description="Allowing column to be skipped if it's missing in the dataset schema.")

class TransformationPlan(BaseModel):
    operations: List[TransformationStep] = Field(description="Ordered list of transformations required to convert the dataset to canonical format")

class CodeValidationResult(BaseModel):
    valid: bool = Field(default=False, description="Final output whether is the code is valid.")
    missing_operations: List[str] = Field(default=[], description="List of operations that are missing in the code." )
    incorrect_operations: List[str] = Field(default=[], description="List of incorrect operations in the code." )
    reasoning: str = Field(description="The reasoning behind the validation output.")

In [ ]:
from typing import TypedDict

class GraphState(TypedDict):
    file_path: str
    schema: dict
    code: str 
    feedback: CodeValidationResult
    transformation_plan: TransformationPlan
    transformed_df: pd.DataFrame
    execution_error: str
    validation_passed: bool 
    validator_feedback: str

In [ ]:
canonical_feature_set = [
    {
        "name": "event_id",
        "description": "Unique identifier for a sales record. Can represent a transaction line item or an aggregated sales record (e.g., store-item-date).",
        "expected_dtype": "string",
        "required": False,
        "generation_strategy": "concatenate_keys_if_missing",
        "possible_source_names": [
            "transaction_id",
            "invoice",
            "invoiceno",
            "orderid",
            "sales_id",
            "billno",
            "trx_no"
        ]
    },
    {
        "name": "item_id",
        "description": "Unique identifier representing the product or SKU.",
        "expected_dtype": "string",
        "required": True,
        "possible_source_names": [
            "stockcode",
            "productid",
            "sku",
            "itemcode",
            "materialno",
            "item_nbr"
        ]
    },
    {
        "name": "location_id",
        "description": "Identifier for store, warehouse, or sales location.",
        "expected_dtype": "string",
        "required": False,
        "possible_source_names": [
            "store",
            "store_nbr",
            "shop_id",
            "location",
            "branch"
        ]
    },
    {
        "name": "quantity",
        "description": "Number of units sold. Can represent per-transaction quantity or aggregated quantity over a time period.",
        "expected_dtype": "numeric",
        "required": True,
        "constraints": {
            "must_be_numeric": True
        },
        "possible_source_names": [
            "qty",
            "quantity",
            "units",
            "unitssold",
            "qty_sold",
            "amount",
            "unit_sales"
        ]
    },
    {
        "name": "event_timestamp",
        "description": "Timestamp when the sales event occurred (transaction time or aggregated date).",
        "expected_dtype": "datetime",
        "required": True,
        "constraints": {
            "must_be_datetime_compatible": True
        },
        "possible_source_names": [
            "invoicedate",
            "ordertime",
            "transactiondate",
            "timestamp",
            "datetime",
            "salesdate",
            "date"
        ]
    },
    {
        "name": "unit_price",
        "description": "Price per unit of the product. May be missing in aggregated datasets.",
        "expected_dtype": "numeric",
        "required": False,
        "constraints": {
            "must_be_numeric": True,
            "must_be_positive": True
        },
        "possible_source_names": [
            "price",
            "unitprice",
            "sellingprice",
            "rate",
            "cost"
        ]
    },
    {
        "name": "total_value",
        "description": "Total sales value for the record (quantity * unit_price).",
        "expected_dtype": "numeric",
        "required": False,
        "derivable": True,
        "derivation_logic": "quantity * unit_price"
    },
    {
        "name": "customer_id",
        "description": "Unique identifier representing the customer.",
        "expected_dtype": "string",
        "required": False,
        "possible_source_names": [
            "customerid",
            "client",
            "buyerid",
            "accountno",
            "partycode"
        ]
    },
    {
        "name": "region",
        "description": "Geographic region where the sale occurred.",
        "expected_dtype": "string",
        "required": False,
        "possible_source_names": [
            "country",
            "region",
            "state",
            "zone",
            "market",
            "city"
        ]
    },
    {
        "name": "promotion_flag",
        "description": "Indicates whether the item was under promotion during the sale.",
        "expected_dtype": "boolean",
        "required": False,
        "possible_source_names": [
            "onpromotion",
            "promo",
            "is_promo",
            "promotion"
        ]
    }
]

In [ ]:
def get_schema_node(state: GraphState):
    file_path = state.get('file_path', None)
    schema = {}
    if file_path:
        df = pd.read_csv(file_path)
        schema = {
            "num_rows": df.shape[0],
            "num_columns": df.shape[1],
            "columns": {}
        }
        for col in df.columns:
            col_data = df[col]
            if pd.api.types.is_numeric_dtype(col_data):
                col_type = "numeric"
            elif pd.api.types.is_datetime64_any_dtype(col_data):
                col_type = "datetime"
            elif pd.api.types.is_bool_dtype(col_data):
                col_type = "boolean"
            else:
                col_type = "categorical"
            col_info = {
                "dtype": str(col_data.dtype),
                "missing_values": int(col_data.isna().sum()),
                "missing_percentage": float(col_data.isna().mean()),
                "unique_values": int(col_data.nunique()),
                "sample_values": col_data.dropna().unique()[:3].tolist(),
                "semantic_type": col_type
            }
            schema["columns"][col] = col_info

    return {"schema": schema}


In [ ]:
TRANSFORMATION_PLANNER_PROMPT = """
You are a senior data engineer responsible for standardizing datasets.

Your task is to analyze a dataset schema and generate a transformation plan
that converts the dataset into a canonical feature set.

The output MUST follow the Pydantic schema for TransformationPlan.

--------------------------------

DATASET SCHEMA

{dataset_schema}

--------------------------------

CANONICAL FEATURE SET

Each canonical feature represents a business concept required by the analytics system.

For each feature you are given:
- feature name
- description
- expected datatype

{canonical_features}

--------------------------------

TRANSFORMATION GOAL

Generate a TransformationPlan that converts the dataset into the canonical feature set.

The transformation may include:

• Renaming columns
• Casting datatypes
• Converting timestamps
• Filling missing values
• Dropping unused columns
• Feature engineering from multiple columns

--------------------------------

RULES

1. Use only columns present in the dataset schema unless creating a new feature via feature engineering.

2. If a canonical feature can be mapped directly, use a rename operation.

3. If a datatype does not match the canonical requirement, add a cast operation.

4. If a timestamp column exists but is not datetime type, use to_datetime and set target_dtype.

5. If a canonical feature can be derived from multiple columns, use feature_engineering.

6. If a canonical feature is marked as required but does not exist in the dataset:
   - Create it using feature_engineering if possible
   - Otherwise, generate it using available columns (e.g., concatenation)
   - Set allow_if_missing = true if generation depends on optional columns

7. If a canonical feature is optional and cannot be mapped or derived:
   - Skip it (do not include unnecessary operations)

8. Do NOT force mappings for columns that do not semantically match the canonical feature.

9. The operations list should be ordered logically.

10. Operations must be executable sequentially. If a column is renamed, all later operations must reference the new column name.

11. Do NOT drop columns unless they are completely irrelevant or duplicate.

12. Additional columns not part of the canonical feature set should be preserved, as they may be useful for downstream tasks such as feature engineering, analytics, or modeling.

13. Only drop columns if they are clearly unnecessary or cannot be used meaningfully.

--------------------------------

OUTPUT FORMAT

Return ONLY valid JSON that matches the following structure:

{transformation_plan}

Do NOT include explanations.
Do NOT include markdown.
Return ONLY valid JSON.
"""

In [ ]:
TRANSFORMATION_REPLANNER_PROMPT = """
You are a senior data engineer responsible for correcting a transformation plan.
The previous transformation plan was executed but failed validation.
Your task is to adjust the plan so that the resulting dataset satisfies the canonical feature set.
The output MUST follow the Pydantic schema for TransformationPlan.

--------------------------------

CANONICAL FEATURE SET

{canonical_features}

--------------------------------

PREVIOUS TRANSFORMATION PLAN

{previous_plan}

--------------------------------

POST VALIDATION ERRORS

The following structural validation errors occurred after executing the plan:

{validator_errors}

--------------------------------

TASK

Modify the previous transformation plan so that the validation errors are resolved.

Only make changes that are necessary to fix the errors.

--------------------------------

RULES

1. Preserve correct operations from the previous plan whenever possible.

2. Only modify or add steps necessary to resolve the validation errors.

3. Ensure all required canonical features are produced:
   - If missing, create them using feature_engineering when possible.

4. If a required feature cannot be directly mapped:
   - Derive or generate it using available columns (e.g., concatenation).

5. If a canonical feature is optional and cannot be mapped or derived:
   - Skip it (do not introduce unnecessary operations).

6. Ensure datatypes match the expected canonical datatype.

7. Operations must be logically ordered and executable sequentially.

8. If a column is renamed, all later operations must reference the new column name.

9. Use only columns present in the dataset schema unless performing feature engineering.

10. When creating or deriving features, prefer simple deterministic formulas (e.g., concat, arithmetic).

11. Use allow_if_missing = true for feature_engineering steps that depend on columns that may not always exist.

12. Do NOT add data cleaning logic such as filtering negative values or removing rows.

13. Return a complete TransformationPlan.

--------------------------------

OUTPUT FORMAT

Return ONLY valid JSON matching the TransformationPlan schema.

{transformation_plan}

Do NOT include explanations.
Do NOT include markdown.
Return ONLY valid JSON.
"""

In [ ]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import SystemMessage, HumanMessage

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
def transformation_planner_node(state: GraphState):
    schema = state.get('schema', {})
    validator_feedback = state.get('validator_feedback', None)
    previous_plan = state.get("transformation_plan", None)
    errors = validator_feedback.get("errors") if validator_feedback else None

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        api_key=GOOGLE_API_KEY
    )

    if errors and previous_plan:
        prompt_text = TRANSFORMATION_REPLANNER_PROMPT.format(
            canonical_features=canonical_feature_set,
            previous_plan=previous_plan,
            validation_errors=errors,
            transformation_plan=TransformationPlan.model_json_schema()
        )
    else:
        prompt_text = TRANSFORMATION_PLANNER_PROMPT.format(
            dataset_schema=schema,
            canonical_features=canonical_feature_set,
            transformation_plan=TransformationPlan.model_json_schema()
        )

    structured_llm = llm.with_structured_output(TransformationPlan, method='function_calling')
    response = structured_llm.invoke([HumanMessage(content=prompt_text)])
    return {"transformation_plan": response.model_dump()}

In [ ]:
CODE_GENERATOR_PROMPT = """
Generate Python pandas code that performs the following transformation plan.
Rules:
- Assume dataframe is named 'df' 
- Perform operations sequentially in the same order as the plan.
- Return ONLY Python code.

Transformation Plan:
{transformation_plan}
"""

CODE_GENERATOR_RETRY_PROMPT = """
The previous code failed.
Transformation Plan:
{transformation_plan}

Previous Code:
{code}

Feedback:
{feedback}

Fix the code so the transformation executes successfully.
Return ONLY Python code.
"""

In [ ]:
def code_generator_node(state: GraphState):
    plan = state.get('transformation_plan', [])
    feedback = state.get('feedback', None)
    existing_code = state.get('code', None)

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        api_key=GOOGLE_API_KEY
    )
    if existing_code and feedback:
        prompt_text = CODE_GENERATOR_RETRY_PROMPT.format(
            transformation_plan = plan,
            code=existing_code,
            feedback=feedback
        )
    else:
        prompt_text = CODE_GENERATOR_PROMPT.format(
            transformation_plan=plan
        )
    response = llm.invoke([HumanMessage(content=prompt_text)])

    return {"code": response.content}

In [ ]:
CODE_VALIDATOR_PROMPT = """
You are a senior data engineer validating transformation code.

Your job is to check whether the generated pandas code correctly implements the provided transformation plan.

You are NOT executing the code.
You are only verifying that the intent of each operation is implemented.

Transformation Plan:
{plan}

Generated Code:
{code}

Validation Rules:
- Every operation in the plan must appear in the code.
- The correct source and target columns must be used.
- Column renames must match exactly.
- Type conversions must match the target dtype.
- Dropped columns must be removed.

Return your answer as JSON with this structure:
{code_validation_result}
"""

In [ ]:
def code_validator_node(state: GraphState):
    plan = state.get('transformation_plan', [])
    code = state.get('code', None)

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        api_key=GOOGLE_API_KEY
    )

    structured_llm = llm.with_structured_output(CodeValidationResult, method='function_calling')
    messages = [
        HumanMessage(content=CODE_VALIDATOR_PROMPT.format(
            plan=plan,
            code=code,
            code_validation_result=CodeValidationResult.model_json_schema()
        ))
    ]
    response = structured_llm.invoke(messages)
    return {"feedback": response.model_dump()}

In [ ]:
from typing import Literal


def code_validation_routing(state: GraphState) -> Literal["code_generator_node", "executor_node"]:
    feedback = state.get('feedback', {})
    if feedback["valid"]:
        return "executor_node"
    else:
        return "code_generator_node"

In [ ]:
import re
def clean_code(code: str) -> str:
    code = code.strip()
    
    code = re.sub(r'^```(?:\w+)?\s*\n?', '', code)
    code = re.sub(r'\n?```$', '', code)
    
    return code.strip()

In [ ]:
import traceback
def executor_node(state: GraphState):
    file_path = state.get('file_path', '')
    code = state.get('code', '')
    try:
        df = pd.read_csv(file_path)

        local_vars = {
            "df": df,
        }
        code = clean_code(code)
        exec(code, {}, local_vars)
        transformed_df = local_vars["df"]
        return {"transformed_df": transformed_df, "execution_error": None}
    except Exception:
        return {"execution_error": traceback.format_exc()}

In [ ]:
def post_validator_node(state: GraphState):
    df = state.get("transformed_df", None)
    errors = []
    warnings = []

    if df is None or df.empty: 
        errors = ["Error in code execution, transformed df not generated."]
        return {
            "validation_passed": False,
            "validation_feedback": {"errors": errors},
            "warnings": None
        }

    # Normalize column names for safety
    df.columns = df.columns.str.strip()

    for feature in canonical_feature_set:
        name = feature["name"]
        required = feature.get("required", False)
        expected_dtype = feature.get("expected_dtype")
        constraints = feature.get("constraints", {})

        # -----------------------------
        # 1. Required column check
        # -----------------------------
        if required and name not in df.columns:
            errors.append(f"Missing required column: {name}")
            continue

        # Skip validation if optional column not present
        if name not in df.columns:
            continue

        col = df[name]

        # -----------------------------
        # 2. Expected dtype validation
        # -----------------------------
        if expected_dtype == "string":
            if not pd.api.types.is_string_dtype(col) and not pd.api.types.is_object_dtype(col):
                errors.append(f"{name} should be string dtype")

        elif expected_dtype == "numeric":
            if not pd.api.types.is_numeric_dtype(col):
                errors.append(f"{name} should be numeric dtype")

        elif expected_dtype == "datetime":
            if not pd.api.types.is_datetime64_any_dtype(col):
                errors.append(f"{name} should be datetime dtype")

        # -----------------------------
        # 3. Constraint checks
        # -----------------------------
        if constraints.get("must_be_numeric"):
            if not pd.api.types.is_numeric_dtype(col):
                errors.append(f"{name} must be numeric")

        if constraints.get("must_be_positive"):
            if (col < 0).any():
                warnings.append(f"{name} contains negative values")

        if constraints.get("must_be_datetime_compatible"):
            if not pd.api.types.is_datetime64_any_dtype(col):
                errors.append(f"{name} must be datetime compatible")

    # -----------------------------
    # Final validation result
    # -----------------------------
    validation_passed = len(errors) == 0

    return {
        "validation_passed": validation_passed,
        "validator_feedback": {"errors": errors if errors else None, "warnings": warnings if warnings else None}
    }

In [ ]:
from langgraph.graph import END
def post_validation_routing(state: GraphState) -> Literal["transformation_planner_node", END]:
    final_validation = state.get("validation_passed", False)
    if final_validation:
        return END
    else:
        return "transformation_planner_node"

In [ ]:
from langgraph.graph import StateGraph, END
graph_builder = StateGraph(GraphState)
graph_builder.add_node("get_schema_node", get_schema_node)
graph_builder.add_node("transformation_planner_node", transformation_planner_node)
graph_builder.add_node("code_generator_node", code_generator_node)
graph_builder.add_node("code_validator_node", code_validator_node)
graph_builder.add_node("executor_node", executor_node)
graph_builder.add_node("post_validator_node", post_validator_node)
graph_builder.set_entry_point("get_schema_node")
graph_builder.add_edge("get_schema_node", "transformation_planner_node")
graph_builder.add_edge("transformation_planner_node", "code_generator_node")
graph_builder.add_edge("code_generator_node", "code_validator_node")
graph_builder.add_conditional_edges("code_validator_node", code_validation_routing)
graph_builder.add_edge("executor_node", "post_validator_node")
graph_builder.add_conditional_edges("post_validator_node", post_validation_routing)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
input_dict = {
    "file_path": 'datasets/sample_dataset.csv',
}
result = graph.invoke(input=input_dict)

In [ ]:
result["schema"]

In [ ]:
result["transformation_plan"]

In [ ]:
result["code"]

In [ ]:
result["feedback"]

In [ ]:
result.keys()

In [ ]:
df = result["transformed_df"]
df.head()

In [ ]:
result["execution_error"]

In [ ]:
result["validation_passed"]

In [ ]:
result["validator_feedback"]